# NB02: Train + Evaluate
**Kaggle: Internet OFF, GPU ON**

Flow: `CONFIG -> SETUP -> DATA -> MODEL -> TRAIN -> CURVES -> EVALUATE(seq6,7) -> VISUALIZE -> SUMMARY`

### Data Split (theo paper):
- **Train**: seq1, seq3, seq4, seq5, seq8, seq9 → 8677 images
- **Valid**: seq2 → 1501 images
- **Test**: seq6, seq7 → 2873 images

### Chuan bi:
1. Add Data: `uav-seg-offline-assets` (NB01 output) + `uav-forest-sunny` (seq1-9)
2. Chinh CONFIG cell (MODEL_NAME, ENCODER, SEED)
3. Run All

In [ ]:
# ============================================================
# CONFIG — CHI THAY DOI O DAY
# ============================================================
# Experiment matrix (15 configs):
#   A1: unet+resnet34       A2: unet+resnet101
#   A3: deeplabv3plus+resnet101  A4: unetpp+resnet50
#   B1: unet+efficientnet-b5  B2: upernet+tu-convnext_base
#   B3: upernet+tu-convnext_large
#   C1/D3: segformer+mit_b3  C2: segformer+mit_b5
#   C3: upernet+swinv2_base  C4: upernet+swinv2_large
#   D1: unet+mit_b3  D2: deeplabv3plus+mit_b3  D4: upernet+mit_b3
# ============================================================

# === Chon experiment ===
MODEL_NAME = 'unet'           # unet | unetpp | deeplabv3plus | upernet | segformer
ENCODER    = 'resnet34'       # resnet34 | resnet50 | resnet101 | efficientnet-b5
                               # mit_b2 | mit_b3 | mit_b5
                               # tu-convnext_base | tu-convnext_large
                               # tu-swinv2_base_window12to24_192to384
                               # tu-swinv2_large_window12to24_192to384
SEED       = 42               # 42 | 123 | 456

# === Hyperparams (FIXED cho tat ca experiments) ===
IMG_SIZE    = (768, 768)
EPOCHS      = 100
PATIENCE    = 15
SAVE_TOP_K  = 2
USE_AMP     = True
NUM_WORKERS = 4
FOCAL_GAMMA = 2.0             # Focal loss gamma (0 = standard CE)

# === Auto LR/BS theo encoder type ===
_TRANSFORMER_ENC = ['mit_b2','mit_b3','mit_b5',
                     'tu-swinv2_base_window12to24_192to384',
                     'tu-swinv2_large_window12to24_192to384']
_CONVNEXT_ENC = ['tu-convnext_base','tu-convnext_large']

if ENCODER in _TRANSFORMER_ENC:
    LR, WEIGHT_DECAY, BATCH_SIZE, WARMUP_EPOCHS = 6e-5, 0.01, 8, 10
elif ENCODER in _CONVNEXT_ENC:
    LR, WEIGHT_DECAY, BATCH_SIZE, WARMUP_EPOCHS = 5e-4, 0.01, 12, 5
else:  # CNN (ResNet, EfficientNet)
    LR, WEIGHT_DECAY, BATCH_SIZE, WARMUP_EPOCHS = 1e-3, 1e-4, 16, 0

# === Resume (de trong neu chay lan dau) ===
RESUME_FROM = ''

print(f'Experiment: {MODEL_NAME} + {ENCODER} | seed={SEED}')
print(f'LR={LR} | WD={WEIGHT_DECAY} | BS={BATCH_SIZE} | IMG={IMG_SIZE}')
print(f'Warmup={WARMUP_EPOCHS}ep | Focal gamma={FOCAL_GAMMA}')

In [ ]:
# ============================================================
# SETUP — packages, cache, seeds, directories
# ============================================================
import os, re, sys, time, math, csv, json, random, shutil, glob
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle')

# --- Install packages offline ---
if IS_KAGGLE:
    for dirpath, _, _ in os.walk('/kaggle/input'):
        if os.path.basename(dirpath) == 'packages':
            if glob.glob(os.path.join(dirpath, '*.whl')):
                os.system(f'pip install -q --no-index --find-links={dirpath} '
                          f'segmentation-models-pytorch albumentations')
                print(f'[OK] Installed from: {dirpath}')
                break
    # Restore pretrained weights cache
    os.environ['HF_HUB_OFFLINE'] = '1'
    os.environ['TRANSFORMERS_OFFLINE'] = '1'
    CACHE_HOME = Path(os.path.expanduser('~/.cache'))
    for dirpath, _, _ in os.walk('/kaggle/input'):
        if os.path.basename(dirpath) == 'dot_cache':
            print(f'Restoring cache from {dirpath}...')
            for src in Path(dirpath).rglob('*'):
                if not src.is_file(): continue
                dst = CACHE_HOME / src.relative_to(dirpath)
                dst.parent.mkdir(parents=True, exist_ok=True)
                if not dst.exists():
                    try: os.symlink(src, dst)
                    except OSError: shutil.copy2(src, dst)
            print('  [OK] Cache restored')
            break
else:
    os.system('pip install -q segmentation-models-pytorch albumentations')

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from PIL import Image
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# --- Seeds ---
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
RUN_NAME = f'{MODEL_NAME}_{ENCODER}_seed{SEED}'

# --- Directories ---
if IS_KAGGLE:
    OUTPUT_DIR = Path('/kaggle/working')
    CKPT_DIR   = OUTPUT_DIR / 'checkpoints' / RUN_NAME
    EVAL_DIR   = OUTPUT_DIR / 'eval_results'
else:
    OUTPUT_DIR = Path('outputs')
    CKPT_DIR   = OUTPUT_DIR / 'checkpoints' / RUN_NAME
    EVAL_DIR   = OUTPUT_DIR / 'eval_results'
CKPT_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = CKPT_DIR / 'train_log.csv'

print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
print(f'Run: {RUN_NAME}')
print(f'Checkpoints: {CKPT_DIR}')

In [ ]:
# ============================================================
# DATA — load sequences, split, augmentation
# ============================================================
NUM_CLASSES = 11
LABEL_COLORS = [
    (0,255,255),(0,127,0),(19,132,69),(0,53,65),(130,76,0),
    (152,251,152),(151,126,171),(250,150,0),(115,176,195),(123,123,123),(0,0,0)
]
CLASS_NAMES = [
    'Sky','Deciduous_trees','Coniferous_trees','Fallen_trees','Dirt_ground',
    'Ground_vegetation','Rocks','Building','Fence','Car','Empty'
]
PALETTE = np.array(LABEL_COLORS, dtype=np.uint8)

# Fast color LUT (O(1) per pixel)
COLOR_LUT = np.full((256,256,256), NUM_CLASSES-1, dtype=np.int64)
for i, (r,g,b) in enumerate(LABEL_COLORS):
    COLOR_LUT[r,g,b] = i

def rgb_to_mask(rgb):
    return COLOR_LUT[rgb[:,:,0], rgb[:,:,1], rgb[:,:,2]]

def mask_to_rgb(mask):
    h, w = mask.shape
    out = np.zeros((h, w, 3), dtype=np.uint8)
    for i in range(NUM_CLASSES):
        out[mask == i] = PALETTE[i]
    return out

class ForestDataset(Dataset):
    def __init__(self, root, sequences, transform=None):
        self.transform = transform
        self.samples = []
        root = Path(root)
        for seq in sequences:
            cd = root / seq / 'color'
            if not cd.exists():
                print(f'  [SKIP] {seq} — not found')
                continue
            for f in sorted(cd.glob('*.png')):
                lf = root / seq / 'labels' / f.name
                if lf.exists():
                    self.samples.append((f, lf))
        print(f'  {len(self.samples)} samples from {sequences}')

    def __len__(self): return len(self.samples)

    def __getitem__(self, i):
        img  = np.array(Image.open(self.samples[i][0]).convert('RGB'))
        mask = rgb_to_mask(np.array(Image.open(self.samples[i][1]).convert('RGB')))
        if self.transform:
            t = self.transform(image=img, mask=mask.astype(np.int64))
            img, mask = t['image'], t['mask']
        return img.float(), mask.long()

# --- Locate sequences ---
if IS_KAGGLE:
    DATA_ROOT = Path('/kaggle/working/data')
    DATA_ROOT.mkdir(exist_ok=True)
    for dirpath, _, _ in os.walk('/kaggle/input', followlinks=True):
        if os.path.basename(dirpath) == 'color':
            seq_dir  = os.path.dirname(dirpath)
            seq_name = os.path.basename(seq_dir)
            if re.match(r'seq\d+$', seq_name):
                link = DATA_ROOT / seq_name
                if not link.exists():
                    os.symlink(seq_dir, link)
else:
    DATA_ROOT = Path('data/forest_sunny')

seqs = sorted([d.name for d in DATA_ROOT.iterdir()
               if d.is_dir() and d.name.startswith('seq')])
print(f'Found {len(seqs)} sequences: {seqs}')
if not seqs:
    raise RuntimeError('No sequences found! Check Add Data.')

# === Paper split: train=seq1,3,4,5,8,9 | val=seq2 | test=seq6,7 ===
TEST_SEQS = ['seq6', 'seq7']   # 2873 images
VAL_SEQS  = ['seq2']           # 1501 images
train_seqs = sorted([s for s in seqs if s not in TEST_SEQS + VAL_SEQS])
print(f'  Train: {train_seqs}')
print(f'  Val:   {VAL_SEQS}')
test_found = [s for s in TEST_SEQS if s in seqs]
print(f'  Test:  {TEST_SEQS} ({len(test_found)}/{len(TEST_SEQS)} found)')

# --- Augmentation ---
IMG_H, IMG_W = IMG_SIZE
train_tf = A.Compose([
    A.Resize(IMG_H, IMG_W),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.Rotate(limit=30, p=0.4, border_mode=0),
    A.RandomBrightnessContrast(0.2, 0.2, p=0.4),
    A.HueSaturationValue(10, 20, 20, p=0.3),
    A.GaussNoise(p=0.2),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2()
])
val_tf = A.Compose([
    A.Resize(IMG_H, IMG_W),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2()
])

train_ds = ForestDataset(DATA_ROOT, train_seqs, train_tf)
val_ds   = ForestDataset(DATA_ROOT, VAL_SEQS, val_tf)
if len(train_ds) == 0:
    raise RuntimeError(f'Train empty! Seqs: {train_seqs}')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
print(f'\nTrain: {len(train_ds)} ({len(train_loader)} batches) | Val: {len(val_ds)}')

In [ ]:
# ============================================================
# CLASS WEIGHTS — sample 200 images to compute frequency
# ============================================================
print('Computing class weights...')
n_sample = min(200, len(train_ds))
counts = np.zeros(NUM_CLASSES, dtype=np.int64)
for idx in np.random.choice(len(train_ds), n_sample, replace=False):
    _, mask = train_ds[idx]
    for c in range(NUM_CLASSES):
        counts[c] += (mask == c).sum().item()

freq = counts / counts.sum()
for i, name in enumerate(CLASS_NAMES):
    print(f'  {name:20s} {freq[i]*100:6.2f}%')

# Inverse frequency, clipped
w = np.ones(NUM_CLASSES, dtype=np.float32)
nz = freq > 0
w[nz] = 1.0 / (freq[nz] * NUM_CLASSES)
w = np.clip(w, 0.5, 10.0)
w = w / w.sum() * NUM_CLASSES
CLASS_WEIGHTS = torch.tensor(w, device=DEVICE)
print(f'\nWeights: {[round(x, 2) for x in w.tolist()]}')

In [ ]:
# ============================================================
# MODEL + LOSS + OPTIMIZER
# ============================================================
MODELS = {
    'unet':          lambda: smp.Unet(encoder_name=ENCODER, encoder_weights='imagenet',
                                       in_channels=3, classes=NUM_CLASSES),
    'unetpp':        lambda: smp.UnetPlusPlus(encoder_name=ENCODER, encoder_weights='imagenet',
                                               in_channels=3, classes=NUM_CLASSES),
    'deeplabv3plus': lambda: smp.DeepLabV3Plus(encoder_name=ENCODER, encoder_weights='imagenet',
                                                in_channels=3, classes=NUM_CLASSES),
    'upernet':       lambda: smp.UPerNet(encoder_name=ENCODER, encoder_weights='imagenet',
                                          in_channels=3, classes=NUM_CLASSES),
    'segformer':     lambda: smp.Segformer(encoder_name=ENCODER, encoder_weights='imagenet',
                                            in_channels=3, classes=NUM_CLASSES),
}
if MODEL_NAME not in MODELS:
    raise ValueError(f'Unknown model: {MODEL_NAME}. Choose from: {list(MODELS.keys())}')

model = MODELS[MODEL_NAME]().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model: {MODEL_NAME} + {ENCODER} | {n_params:,} params ({n_params/1e6:.1f}M)')

# --- Focal + Dice Loss (better for imbalanced classes) ---
class FocalDiceLoss(nn.Module):
    def __init__(self, nc, class_weights=None, gamma=2.0):
        super().__init__()
        self.nc = nc
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss(weight=class_weights, reduction='none')

    def forward(self, logits, targets):
        # Focal CE
        ce = self.ce(logits, targets)
        if self.gamma > 0:
            pt = torch.exp(-ce)
            ce_term = ((1 - pt) ** self.gamma * ce).mean()
        else:
            ce_term = ce.mean()
        # Dice
        probs = F.softmax(logits, dim=1)
        tgt = F.one_hot(targets.clamp(0, self.nc-1), self.nc).permute(0,3,1,2).float()
        inter = (probs * tgt).sum(dim=(0,2,3))
        card  = probs.sum(dim=(0,2,3)) + tgt.sum(dim=(0,2,3))
        dice  = 1.0 - ((2 * inter + 1e-6) / (card + 1e-6)).mean()
        return ce_term + 0.5 * dice

criterion = FocalDiceLoss(NUM_CLASSES, CLASS_WEIGHTS, gamma=FOCAL_GAMMA)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# Warmup + Cosine schedule
def lr_lambda(ep):
    if ep < WARMUP_EPOCHS:
        return (ep + 1) / max(1, WARMUP_EPOCHS)
    t = (ep - WARMUP_EPOCHS) / max(1, EPOCHS - WARMUP_EPOCHS)
    return 0.5 * (1 + math.cos(math.pi * t))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = GradScaler('cuda', enabled=USE_AMP and DEVICE == 'cuda')

print(f'Loss: FocalDice (gamma={FOCAL_GAMMA})')
print(f'Optimizer: AdamW LR={LR} WD={WEIGHT_DECAY}')
print(f'Scheduler: {"Warmup(" + str(WARMUP_EPOCHS) + "ep)+" if WARMUP_EPOCHS else ""}Cosine')

In [ ]:
# ============================================================
# RESUME (if checkpoint exists)
# ============================================================
start_epoch = 1
best_miou   = 0.0
saved       = []  # list of (miou, path)

if RESUME_FROM:
    resume_path = Path(RESUME_FROM)
    if resume_path.exists():
        print(f'Resuming from: {resume_path.name}')
        ckpt = torch.load(resume_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        if 'scheduler_state_dict' in ckpt:
            scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        if 'scaler_state_dict' in ckpt:
            scaler.load_state_dict(ckpt['scaler_state_dict'])
        start_epoch = ckpt.get('epoch', 0) + 1
        best_miou   = ckpt.get('metric', ckpt.get('miou', 0.0))
        print(f'  Resumed: epoch={start_epoch-1}, best_mIoU={best_miou:.4f}')
    else:
        print(f'[WARN] Resume path not found: {resume_path}')

print(f'Start epoch: {start_epoch} | Best mIoU: {best_miou:.4f}')

In [ ]:
# ============================================================
# TRAIN LOOP
# ============================================================
# Init log
if not LOG_FILE.exists():
    with open(LOG_FILE, 'w', newline='') as f:
        csv.writer(f).writerow(['epoch','train_loss','val_loss','mIoU','pixel_acc','lr','time_s'])

history = {'train_loss':[], 'val_loss':[], 'miou':[], 'pixel_acc':[], 'lr':[]}
no_improve = 0

for epoch in range(start_epoch, EPOCHS + 1):
    t0 = time.time()

    # ---- Train ----
    model.train()
    train_loss, n_batch = 0.0, 0
    for imgs, masks in tqdm(train_loader, desc=f'E{epoch:03d} train', leave=False):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        with autocast('cuda', enabled=USE_AMP and DEVICE == 'cuda'):
            loss = criterion(model(imgs), masks)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()
        n_batch += 1
    train_loss /= n_batch

    # ---- Validate ----
    model.eval()
    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    val_loss, n_val = 0.0, 0
    with torch.no_grad():
        for imgs, masks in tqdm(val_loader, desc=f'E{epoch:03d} val', leave=False):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            with autocast('cuda', enabled=USE_AMP and DEVICE == 'cuda'):
                out = model(imgs)
                val_loss += criterion(out, masks).item()
                n_val += 1
            preds = out.argmax(1).cpu().numpy()
            gts   = masks.cpu().numpy()
            for p, g in zip(preds, gts):
                v = (g >= 0) & (g < NUM_CLASSES)
                np.add.at(cm, (g[v], p[v]), 1)
    val_loss /= n_val

    # Metrics
    inter = np.diag(cm)
    union = cm.sum(1) + cm.sum(0) - inter
    iou   = np.where(union > 0, inter / union, np.nan)
    miou  = float(np.nanmean(iou))
    pa    = float(inter.sum() / (cm.sum() + 1e-9))

    scheduler.step()
    lr = optimizer.param_groups[0]['lr']
    elapsed = time.time() - t0

    # Log
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['miou'].append(miou)
    history['pixel_acc'].append(pa)
    history['lr'].append(lr)

    with open(LOG_FILE, 'a', newline='') as f:
        csv.writer(f).writerow([epoch, f'{train_loss:.6f}', f'{val_loss:.6f}',
                                f'{miou:.6f}', f'{pa:.6f}', f'{lr:.2e}', f'{elapsed:.1f}'])

    print(f'E{epoch:03d}/{EPOCHS} | loss {train_loss:.4f}/{val_loss:.4f} | '
          f'mIoU {miou:.4f} | PA {pa:.4f} | lr {lr:.2e} | {elapsed:.0f}s')

    # ---- Checkpoint ----
    if miou > best_miou:
        best_miou  = miou
        no_improve = 0
        ckpt_path  = CKPT_DIR / f'epoch_{epoch:03d}_miou_{miou:.4f}.pth'
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'metric': miou,
            'config': {'model': MODEL_NAME, 'encoder': ENCODER,
                       'seed': SEED, 'img_size': IMG_SIZE}
        }, ckpt_path)

        saved.append((miou, ckpt_path))
        saved.sort(key=lambda x: x[0])
        while len(saved) > SAVE_TOP_K:
            _, old = saved.pop(0)
            if old.exists(): old.unlink()

        print(f'  *** BEST: {miou:.4f} -> {ckpt_path.name}')
        for i, name in enumerate(CLASS_NAMES):
            if not np.isnan(iou[i]) and iou[i] > 0:
                print(f'      {name:20s}: {iou[i]:.4f}')
    else:
        no_improve += 1

    if no_improve >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch} (patience={PATIENCE})')
        break

print(f'\n{"="*60}')
print(f'TRAINING DONE: {RUN_NAME}')
print(f'Best val mIoU: {best_miou:.4f}')
print(f'{"="*60}')

In [ ]:
# ============================================================
# TRAINING CURVES
# ============================================================
with open(CKPT_DIR / 'history.json', 'w') as f:
    json.dump(history, f, indent=2)

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[0].set_xlabel('Epoch')

axes[1].plot(history['miou'], 'g')
axes[1].axhline(best_miou, color='r', ls='--', alpha=0.5, label=f'Best={best_miou:.4f}')
axes[1].set_title('mIoU'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
axes[1].set_xlabel('Epoch')

axes[2].plot(history['pixel_acc'], 'b')
axes[2].set_title('Pixel Accuracy'); axes[2].grid(True, alpha=0.3)
axes[2].set_xlabel('Epoch')

axes[3].plot(history['lr'], 'orange')
axes[3].set_title('Learning Rate'); axes[3].grid(True, alpha=0.3)
axes[3].set_xlabel('Epoch')

fig.suptitle(f'{MODEL_NAME} + {ENCODER} (seed={SEED}) @ {IMG_H}x{IMG_W}', fontweight='bold')
fig.tight_layout()
fig.savefig(EVAL_DIR / f'{RUN_NAME}_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {EVAL_DIR / f"{RUN_NAME}_training_curves.png"}')

---
# EVALUATE ON TEST SET (seq6, seq7)

Model vua train xong — **load best checkpoint roi evaluate**.

Output: `test_results.json` + `results_all.csv` + confusion matrix + qualitative viz

In [ ]:
# ============================================================
# EVALUATE ON TEST SET (seq6, seq7)
# ============================================================

# --- Check test sequences ---
test_ok = all((DATA_ROOT / s / 'color').exists() for s in TEST_SEQS)
if not test_ok:
    print(f'[SKIP] Some test sequences not found — chi co ket qua validation')
    print(f'  Add "uav-forest-sunny" dataset (co seq6,7) vao notebook')
    test_miou = test_pa = test_mf1 = -1
else:
    # --- Load best checkpoint ---
    if saved:
        best_path = saved[-1][1]
        print(f'Loading best checkpoint: {best_path.name}')
        ckpt_data = torch.load(best_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt_data['model_state_dict'])
    else:
        print('[WARN] No checkpoints — using current weights')

    model.eval()

    # --- Model efficiency ---
    params_M = n_params / 1e6
    flops_G  = -1.0
    try:
        from fvcore.nn import FlopCountAnalysis
        dummy = torch.zeros(1, 3, *IMG_SIZE).to(DEVICE)
        fa = FlopCountAnalysis(model, dummy)
        fa.unsupported_ops_warnings(False)
        fa.uncalled_modules_warnings(False)
        flops_G = fa.total() / 1e9
    except Exception:
        print('  [INFO] fvcore not available, skipping FLOPs')

    fps = -1.0
    if DEVICE == 'cuda':
        dummy = torch.zeros(1, 3, *IMG_SIZE).to(DEVICE)
        with torch.no_grad():
            for _ in range(10): model(dummy)
        torch.cuda.synchronize()
        t0 = time.time()
        with torch.no_grad():
            for _ in range(50): model(dummy)
        torch.cuda.synchronize()
        fps = 50.0 / (time.time() - t0)

    print(f'Params: {params_M:.2f}M | FLOPs: {flops_G:.2f}G | FPS: {fps:.1f}')

    # --- Test dataset (seq6 + seq7) ---
    test_ds = ForestDataset(DATA_ROOT, TEST_SEQS, val_tf)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=(DEVICE=='cuda'))

    # --- Evaluate ---
    test_cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    saved_samples = []

    with torch.no_grad():
        for imgs, masks in tqdm(test_loader, desc='Test (seq6,7)'):
            imgs = imgs.to(DEVICE)
            with autocast('cuda', enabled=USE_AMP and DEVICE == 'cuda'):
                out = model(imgs)
            preds = out.argmax(1).cpu().numpy()
            gts   = masks.numpy()

            for p, g in zip(preds, gts):
                v = (g >= 0) & (g < NUM_CLASSES)
                np.add.at(test_cm, (g[v], p[v]), 1)

            # Save first 8 for visualization
            if len(saved_samples) < 8:
                for i in range(min(len(imgs), 8 - len(saved_samples))):
                    img_np = imgs[i].cpu().numpy().transpose(1,2,0)
                    std_  = np.array([0.229, 0.224, 0.225])
                    mean_ = np.array([0.485, 0.456, 0.406])
                    img_np = np.clip(img_np * std_ + mean_, 0, 1)
                    saved_samples.append((img_np, preds[i], gts[i]))

    # --- Metrics ---
    t_inter = np.diag(test_cm)
    t_union = test_cm.sum(1) + test_cm.sum(0) - t_inter
    t_iou   = np.where(t_union > 0, t_inter / t_union, np.nan)
    test_miou = float(np.nanmean(t_iou))
    test_pa   = float(t_inter.sum() / (test_cm.sum() + 1e-9))

    t_prec = np.where(test_cm.sum(0) > 0, t_inter / test_cm.sum(0), np.nan)
    t_rec  = np.where(test_cm.sum(1) > 0, t_inter / test_cm.sum(1), np.nan)
    t_f1   = np.where((t_prec+t_rec) > 0, 2*t_prec*t_rec/(t_prec+t_rec), np.nan)
    test_mf1 = float(np.nanmean(t_f1))

    print(f'\n{"="*60}')
    print(f'TEST RESULTS — {MODEL_NAME} + {ENCODER} (seed={SEED})')
    print(f'{"="*60}')
    print(f'  mIoU:      {test_miou*100:.2f}%')
    print(f'  Pixel Acc: {test_pa*100:.2f}%')
    print(f'  mF1:       {test_mf1*100:.2f}%')
    print(f'  Params:    {params_M:.2f}M')
    print(f'  FLOPs:     {flops_G:.2f}G')
    print(f'  FPS:       {fps:.1f}')
    print(f'\nPer-class IoU:')
    for i, name in enumerate(CLASS_NAMES):
        v = t_iou[i]
        print(f'  {name:20s}: {v*100:.2f}%' if not np.isnan(v) else f'  {name:20s}: N/A')

    # --- Save JSON ---
    test_results = {
        'model': MODEL_NAME, 'encoder': ENCODER, 'seed': SEED,
        'config': f'{MODEL_NAME}_{ENCODER}', 'img_size': list(IMG_SIZE),
        'split': {'train': ['seq1','seq3','seq4','seq5','seq8','seq9'],
                  'val': ['seq2'], 'test': ['seq6','seq7']},
        'best_val_miou': round(best_miou*100, 2),
        'mIoU': round(test_miou*100, 2),
        'PA':   round(test_pa*100, 2),
        'mF1':  round(test_mf1*100, 2),
        'params_M':  round(params_M, 2),
        'flops_G':   round(flops_G, 2),
        'fps':       round(fps, 1),
        'per_class_iou': {CLASS_NAMES[i]: round(float(t_iou[i])*100, 2)
                          for i in range(NUM_CLASSES) if not np.isnan(t_iou[i])},
        'per_class_f1':  {CLASS_NAMES[i]: round(float(t_f1[i])*100, 2)
                          for i in range(NUM_CLASSES) if not np.isnan(t_f1[i])},
    }
    with open(EVAL_DIR / f'{RUN_NAME}_test_results.json', 'w') as f:
        json.dump(test_results, f, indent=2)
    np.save(EVAL_DIR / f'{RUN_NAME}_cm.npy', test_cm)

    # --- Append to CSV ---
    import pandas as pd
    row = {
        'config': f'{MODEL_NAME}_{ENCODER}', 'model': MODEL_NAME, 'encoder': ENCODER, 'seed': SEED,
        'best_val_miou': round(best_miou*100,2),
        'mIoU': round(test_miou*100,2), 'PA': round(test_pa*100,2),
        'mF1': round(test_mf1*100,2), 'params_M': round(params_M,2),
        'flops_G': round(flops_G,2), 'fps': round(fps,1),
    }
    for i, cls in enumerate(CLASS_NAMES):
        row[f'iou_{cls}'] = round(float(t_iou[i])*100,2) if not np.isnan(t_iou[i]) else -1.0

    csv_path = EVAL_DIR / 'results_all.csv'
    df_new = pd.DataFrame([row])
    if csv_path.exists():
        df_old = pd.read_csv(csv_path)
        df_old = df_old[~((df_old['config']==f'{MODEL_NAME}_{ENCODER}') & (df_old['seed']==SEED))]
        df_all = pd.concat([df_old, df_new], ignore_index=True)
    else:
        df_all = df_new
    df_all.to_csv(csv_path, index=False)
    print(f'\nSaved: {csv_path} ({len(df_all)} experiments total)')

In [ ]:
# ============================================================
# VIZ 1: Qualitative Predictions
# ============================================================
if test_ok and saved_samples:
    n_viz = min(6, len(saved_samples))
    fig, axes = plt.subplots(n_viz, 3, figsize=(15, n_viz * 4))
    if n_viz == 1: axes = axes[np.newaxis]

    for i in range(n_viz):
        img, pred, gt = saved_samples[i]
        axes[i,0].imshow(img);           axes[i,0].set_title('Input');        axes[i,0].axis('off')
        axes[i,1].imshow(mask_to_rgb(gt));  axes[i,1].set_title('Ground Truth'); axes[i,1].axis('off')
        axes[i,2].imshow(mask_to_rgb(pred));axes[i,2].set_title('Prediction');   axes[i,2].axis('off')

    patches = [mpatches.Patch(color=np.array(LABEL_COLORS[i])/255, label=CLASS_NAMES[i])
               for i in range(NUM_CLASSES)]
    fig.legend(handles=patches, loc='lower center', ncol=6,
               bbox_to_anchor=(0.5, -0.01), fontsize=9, frameon=True)
    plt.suptitle(f'Qualitative — {RUN_NAME} | mIoU={test_miou*100:.2f}%', fontsize=13, y=1.01)
    plt.tight_layout()
    out = EVAL_DIR / f'{RUN_NAME}_qualitative.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out}')
else:
    print('[SKIP] No test samples for visualization')

In [ ]:
# ============================================================
# VIZ 2: Confusion Matrix
# ============================================================
if test_ok:
    try:
        import seaborn as sns
    except ImportError:
        print('[WARN] seaborn not available — skip heatmap')
        sns = None

    if sns is not None:
        cm_norm = test_cm.astype(float) / (test_cm.sum(1, keepdims=True) + 1e-9)
        fig, ax = plt.subplots(figsize=(12, 10))
        sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                    ax=ax, linewidths=0.5, vmin=0, vmax=1)
        ax.set_xlabel('Predicted', fontsize=12)
        ax.set_ylabel('Ground Truth', fontsize=12)
        ax.set_title(f'Confusion Matrix — {RUN_NAME}', fontsize=13)
        plt.xticks(rotation=40, ha='right', fontsize=9)
        plt.yticks(rotation=0, fontsize=9)
        plt.tight_layout()
        out = EVAL_DIR / f'{RUN_NAME}_confusion_matrix.png'
        plt.savefig(out, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Saved: {out}')
else:
    print('[SKIP] No test data for confusion matrix')

In [ ]:
# ============================================================
# FINAL SUMMARY
# ============================================================
print('=' * 60)
print(f'COMPLETE: {RUN_NAME}')
print('=' * 60)

print(f'  Val mIoU:  {best_miou*100:.2f}%')
if test_ok:
    print(f'  Test mIoU: {test_miou*100:.2f}%')
    print(f'  Test PA:   {test_pa*100:.2f}%')
    print(f'  Test mF1:  {test_mf1*100:.2f}%')
    print(f'  Params:    {params_M:.2f}M | FLOPs: {flops_G:.2f}G | FPS: {fps:.1f}')

# Training log
if LOG_FILE.exists():
    import pandas as pd
    log_df = pd.read_csv(LOG_FILE)
    print(f'\nTraining log: {len(log_df)} epochs')
    print(log_df.tail(5).to_string(index=False))

# List outputs
print(f'\nOutput files:')
for p in sorted(EVAL_DIR.glob('*')):
    if p.is_file():
        sz = p.stat().st_size
        unit = 'KB' if sz < 1024**2 else 'MB'
        val = sz/1024 if sz < 1024**2 else sz/1024**2
        print(f'  {p.name:50s} {val:>7.1f} {unit}')

print(f'\nNEXT STEPS:')
print(f'  1. Click [Save Version] on Kaggle')
print(f'  2. Change MODEL_NAME/ENCODER/SEED and run again')
print(f'  3. After all experiments -> use results_all.csv in NB04')